In [21]:
import sys

import pandas as pd

print(sys.version)
print(pd.__version__)


3.13.15 | packaged by Anaconda, Inc. | (main, Aug 14 2026, 17:19:34) [Clang 20.1.8 ]
3.0.5


In [22]:
listings = pd.read_csv('data/listings_clean.csv')

listings.shape

(4574, 73)

In [23]:
series = pd.Series([3, 5, 8, 10, 12])

print(series.median())
print(series.mean())

8.0
7.6


In [24]:
print(listings['number_of_reviews'].mean().round())
print(listings['number_of_reviews'].median())

36.0
9.0


how many reviews does a typical listing have? 9


In [25]:
print(listings['number_of_reviews'].describe().round(2))


count    4574.00
mean       35.75
std        78.38
min         0.00
25%         2.00
50%         9.00
75%        29.00
max      1293.00
Name: number_of_reviews, dtype: float64


In [26]:
percentiles = listings['number_of_reviews'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
print(percentiles)


0.10      0.00
0.25      2.00
0.50      9.00
0.75     29.00
0.90     94.00
0.99    400.27
Name: number_of_reviews, dtype: float64


In [27]:
q1 = listings['number_of_reviews'].quantile(0.25)
q3 = listings['number_of_reviews'].quantile(0.75)
iqr = q3 - q1

print(f"Q1: {q1:.1f}")
print(f"Q3: {q3:.1f}")
print(f"IQR: {iqr:.1f}")


Q1: 2.0
Q3: 29.0
IQR: 27.0


IQR ligger i ett spann på 27 resencioner när max är 1293

In [28]:
upper_fence = q3 + 1.5 * iqr
lower_fence = q1 - 1.5 * iqr

print(f"Upper fence: {upper_fence:.2f}")
print(f"Lower fence: {lower_fence:.2f}")

# Find outliers
outliers = listings[listings['number_of_reviews'] > upper_fence]
print(f"Number of outliers: {len(outliers)}")


Upper fence: 69.50
Lower fence: -38.50
Number of outliers: 582


In [29]:
listings['review_outlier'] = listings['number_of_reviews'] > upper_fence
print(f"Number of outliers marked: {listings['review_outlier'].sum()}")


Number of outliers marked: 582


In [35]:
print(outliers['room_type'].value_counts())
print(" ")
print(outliers['price'].median())
print(" ")
print(listings[listings['number_of_reviews'] <= upper_fence]['price'].median())



room_type
Entire home/apt    446
Private room       121
Shared room         14
Hotel room           1
Name: count, dtype: int64
 
1459.54
 
1867.0


In [42]:
comparison = pd.DataFrame({
    'All reviews': [listings['number_of_reviews'].count().round(2), 
                   listings['number_of_reviews'].mean().round(2),
                   listings['number_of_reviews'].median().round(2),
                   listings['number_of_reviews'].std().round(2)],
    'Without outliers': [listings[listings['review_outlier'] == False]['number_of_reviews'].count(),
                        listings[listings['review_outlier'] == False]['number_of_reviews'].mean(),
                        listings[listings['review_outlier'] == False]['number_of_reviews'].median(),
                        listings[listings['review_outlier'] == False]['number_of_reviews'].std()]
}, index=['Count', 'Mean', 'Median', 'Standard Deviation']).round(2)

print(comparison)



                    All reviews  Without outliers
Count                   4574.00           3992.00
Mean                      35.75             12.95
Median                     9.00              7.00
Standard Deviation        78.38             15.54


In [47]:
reviews_by_room = listings.groupby('room_type')['number_of_reviews'].describe().round(0)
print(reviews_by_room)

                  count   mean    std   min   25%   50%    75%     max
room_type                                                             
Entire home/apt  3760.0   34.0   76.0   0.0   2.0   8.0   27.0  1293.0
Hotel room          3.0   61.0   16.0  49.0  52.0  56.0   68.0    79.0
Private room      787.0   41.0   84.0   0.0   2.0  12.0   38.0   867.0
Shared room        24.0  114.0  122.0   0.0  28.0  82.0  146.0   510.0


In [48]:
columns = ['number_of_reviews', 'price', 'accommodates', 'bedrooms',
           'availability_365', 'review_scores_rating', 'minimum_nights',
           'days_since_review']


In [49]:
corr_reviews = listings[columns].corr()['number_of_reviews'].round(2)
print(corr_reviews)


number_of_reviews       1.00
price                  -0.09
accommodates           -0.05
bedrooms               -0.14
availability_365        0.18
review_scores_rating   -0.04
minimum_nights         -0.07
days_since_review      -0.19
Name: number_of_reviews, dtype: float64


- days_since_review starkast
- review_scores_rating svagast

In [54]:
corr_all = listings['number_of_reviews'].corr(listings['availability_365'])

# Correlation without the outliers
no_outliers = listings[~listings['review_outlier']]
corr_without = no_outliers['number_of_reviews'].corr(no_outliers['availability_365'])

print(f"With outliers: {corr_all:.4f}")
print(f"Without outliers: {corr_without:.4f}")

With outliers: 0.1832
Without outliers: 0.0022


In [55]:
# With the outliers the correlation is 0.18, which is weak.
# Without them it is 0.0022, so there is no linear connection at all.
# The weak connection came only from the 582 outliers.